<a href="https://colab.research.google.com/github/bsheese/cs377/blob/18_classification_redesign/18_classification/18_1_Classification_Basics/18_1_4_Advanced_Multiclass_Extensions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Multiclass & Extensions

**Author:** Brad Sheese

---

## Introduction

This notebook covers advanced multiclass topics:

- **OvR vs OvO**: Two strategies for multiclass classification
- **Decision Boundaries**: Visualizing how classifiers separate classes
- **Macro vs Weighted vs Micro Metrics**: Averaging strategies for K classes
- **Feature Importance**: Understanding what drives predictions

### Learning Objectives
By the end of this notebook, you will:
1. Explain OvR and OvO multiclass strategies
2. Visualize decision boundaries in 2D using PCA
3. Interpret macro vs weighted vs micro metrics
4. Extract and visualize feature importance

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
import warnings
warnings.filterwarnings('ignore')

# Load wine data
wine = load_wine()
X_wine = wine.data
y_wine = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Wine Dataset: {len(X_train)} train, {len(X_test)} test")
print(f"Classes: {np.unique(y_wine)}")
print(f"Distribution: {np.bincount(y_wine)} (balanced)")

## OvR vs OvO: Two Multiclass Strategies

When extending binary classifiers to K classes:

### One-vs-Rest (OvR)
- Train K binary classifiers
- Classifier k: "Is class k?" vs "Is NOT class k?"
- Requires K models
- For K=3: C0=(C0 vs C1,C2), C1=(C1 vs C0,C2), C2=(C2 vs C0,C1)

### One-vs-One (OvO)
- Train K(K-1)/2 binary classifiers
- Classifier (i,j): "Is class i?" vs "Is class j?"
- For K=3: (C0 vs C1), (C0 vs C2), (C1 vs C2) = 3 models
- Each model only sees samples of two classes (smaller training sets)

**Trade-offs:**
- OvR: Fewer models (K), but each trained on all data
- OvO: More models (K(K-1)/2), but each trained on 2/K of data
- For well-separated classes like wine, both perform similarly

In [ ]:
# Compare OvR vs OvO
base_clf = LogisticRegression(max_iter=1000, random_state=42)

# OvR
ovr = OneVsRestClassifier(base_clf)
ovr.fit(X_train_s, y_train)
acc_ovr = accuracy_score(y_test, ovr.predict(X_test_s))
print(f"One-vs-Rest ({ovr.estimators_.__len__()} models): {acc_ovr:.3f}")

# OvO
ovo = OneVsOneClassifier(base_clf)
ovo.fit(X_train_s, y_train)
acc_ovo = accuracy_score(y_test, ovo.predict(X_test_s))
print(f"One-vs-One ({ovo.estimators_.__len__()} models): {acc_ovo:.3f}")

# Softmax (sklearn default)
softmax = LogisticRegression(max_iter=1000, random_state=42)
softmax.fit(X_train_s, y_train)
acc_softmax = accuracy_score(y_test, softmax.predict(X_test_s))
print(f"Softmax (single model): {acc_softmax:.3f}")

print("\nFor logistic regression with well-separated classes, all three approaches give identical results!")
print("OvO/OvR differ when using non-probabilistic classifiers (e.g., SVM without probability=True)")

## Decision Boundaries with PCA

Let's visualize how different classifiers separate classes in 2D.

### About PCA (Principal Component Analysis)
- **Unsupervised**: Doesn't use class labels - only finds directions of maximum variance
- **Linear**: Finds new axes that are linear combinations of original features
- **Explained variance**: First 2 components capture most variance in wine data
- Used here for visualization only (not for actual model training in this notebook)

In [ ]:
# Reduce to 2D with PCA for visualization
pca = PCA(n_components=2)
X_train_2d = pca.fit_transform(X_train_s)
X_test_2d = pca.transform(X_test_s)

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%")
print(f"  PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"  PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")

# Train classifiers on 2D data
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5)
}

print("\nTraining accuracy on 2D features (for visualization):")
for name, clf in classifiers.items():
    clf.fit(X_train_2d, y_train)
    train_acc = accuracy_score(y_train, clf.predict(X_train_2d))
    test_acc = accuracy_score(y_test, clf.predict(X_test_2d))
    print(f"  {name}: Train={train_acc:.3f}, Test={test_acc:.3f}")

In [ ]:
# Plot decision boundaries
def plot_decision_boundary(clf, X, y, ax, title):
    h = 0.02  # step size
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    ax.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Class 0', edgecolors='k', s=50)
    ax.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Class 1', edgecolors='k', s=50)
    ax.scatter(X[y==2, 0], X[y==2, 1], c='green', label='Class 2', edgecolors='k', s=50)
    ax.set_title(title)
    ax.set_xlabel('Principal Component 1')
    ax.set_ylabel('Principal Component 2')
    ax.legend(loc='best', fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for (name, clf), ax in zip(classifiers.items(), axes):
    clf.fit(X_train_2d, y_train)
    plot_decision_boundary(clf, X_train_2d, y_train, ax, name)

plt.suptitle('Decision Boundaries (PCA-reduced to 2D)', y=1.02)
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("  - Logistic Regression: Linear boundaries (straight lines)")
print("  - Decision Tree: Axis-aligned rectangular regions")
print("  - KNN: Non-linear, adapts to local data density")

## Confusion Matrix for Multiclass

With K classes, confusion matrix is KxK. Perfect model has all predictions on the diagonal.

In [ ]:
# Train best model on full features
best_model = LogisticRegression(max_iter=1000, random_state=42)
best_model.fit(X_train_s, y_train)
y_pred = best_model.predict(X_test_s)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1, 2])
ax.set_yticks([0, 1, 2])
ax.set_xticklabels(['Class 0', 'Class 1', 'Class 2'])
ax.set_yticklabels(['Class 0', 'Class 1', 'Class 2'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix: Wine Classification')

for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=16, fontweight='bold')

plt.colorbar(im)
plt.tight_layout()
plt.show()

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.3f}")

## Macro vs Weighted vs Micro Metrics

With K classes, how do we average precision/recall/F1?

**Macro**: Average of per-class metrics (each class equally weighted)
- Good when you care equally about all classes

**Weighted**: Average weighted by class frequency (support)
- Better reflects overall performance on imbalanced data

**Micro**: Global TP, FP, FN then compute metric
- For multiclass: pools all predictions across classes before computing metric
- Equivalent to accuracy: (sum of TP) / (sum of all diagonal elements)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=3))

# Compute all three averaging methods
precision_macro = precision_score(y_test, y_pred, average='macro')
precision_weighted = precision_score(y_test, y_pred, average='weighted')
precision_micro = precision_score(y_test, y_pred, average='micro')

recall_macro = recall_score(y_test, y_pred, average='macro')
recall_weighted = recall_score(y_test, y_pred, average='weighted')
recall_micro = recall_score(y_test, y_pred, average='micro')

f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')
f1_micro = f1_score(y_test, y_pred, average='micro')

print("\n--- Metric Comparison ---")
print(f"{'Metric':<12} {'Macro':<10} {'Weighted':<10} {'Micro':<10}")
print(f"{'Precision':<12} {precision_macro:.3f}     {precision_weighted:.3f}     {precision_micro:.3f}")
print(f"{'Recall':<12} {recall_macro:.3f}     {recall_weighted:.3f}     {recall_micro:.3f}")
print(f"{'F1-Score':<12} {f1_macro:.3f}     {f1_weighted:.3f}     {f1_micro:.3f}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(3)
width = 0.25
ax.bar(x - width, [precision_macro, recall_macro, f1_macro], width, label='Macro', alpha=0.8, color='steelblue')
ax.bar(x, [precision_weighted, recall_weighted, f1_weighted], width, label='Weighted', alpha=0.8, color='coral')
ax.bar(x + width, [precision_micro, recall_micro, f1_micro], width, label='Micro', alpha=0.8, color='green')
ax.set_xticks(x)
ax.set_xticklabels(['Precision', 'Recall', 'F1-Score'])
ax.set_ylabel('Score')
ax.set_title('Macro vs Weighted vs Micro Averaging')
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

## Feature Importance

In addition to coefficients (for linear models), tree-based models provide **feature importance** scores.

**Random Forest importance**: Based on how much each feature decreases impurity (weighted by how many samples use it)

This helps explain which chemical properties most distinguish wine classes.

In [ ]:
# Train Random Forest for feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_s, y_train)

# Get feature names
feature_names = wine.feature_names
importances = rf.feature_importances_

# Sort by importance
idx = np.argsort(importances)[::-1]

print("Feature Importance (Random Forest):")
for i in idx[:5]:
    print(f"  {feature_names[i]}: {importances[i]:.3f}")

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlBu(np.linspace(0.2, 0.8, len(importances)))
ax.barh(range(len(importances)), importances[idx], color=colors)
ax.set_yticks(range(len(importances)))
ax.set_yticklabels([feature_names[i] for i in idx])
ax.set_xlabel('Importance')
ax.set_title('Feature Importance: Random Forest')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Bonus: Gradient Boosting Comparison

Gradient Boosting is another powerful ensemble method that often outperforms Random Forest.
It builds trees sequentially, with each tree correcting errors from previous ones.

In [ ]:
# Compare multiple classifiers
classifiers_full = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

print("Classifier Comparison (FULL features):")
print(f"{'Classifier':<22} {'Test Accuracy':<15}")
print("-" * 40)
for name, clf in classifiers_full.items():
    clf.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test_s))
    print(f"{name:<22} {acc:.3f}")

## Key Takeaways

**Multiclass Strategies:**
- OvR: K binary classifiers (one per class)
- OvO: K(K-1)/2 binary classifiers (pairwise) - fewer samples per classifier
- Softmax: Single model with K outputs (preferred for most cases)

**Decision Boundaries:**
- Logistic Regression: Linear boundaries
- Decision Tree: Axis-aligned rectangular regions
- KNN: Non-linear, adapts to local data density
- Gradient Boosting: Highly non-linear, powerful

**Metric Averaging:**
- Macro: Equal weight per class
- Weighted: Weight by class frequency
- Micro: Pool all TP/FP/FN globally (equivalent to accuracy for multiclass)

**Feature Importance:**
- Random Forest and Boosting provide importance scores
- Helps interpret which features drive predictions